In [34]:
import re
from typing import List, Dict, Optional
import numpy as np
import pandas as pd

In [35]:
def parse_perf_runs_from_cycles(log_text: str, cpu_hz: Optional[int] = None) -> List[Dict[str, float]]:
    """
    Parse logs with per-stage 'CPU TOTAL: <cycles> cycles' (e.g., Ethos-U prints).
    Returns: list of dicts, one per run, mapping stage -> seconds.

    - Detects CPU clock from: 'Processor internal clock: <Hz>Hz' unless cpu_hz is provided.
    - A run is segmented by 'INFO - Running inference on image'.
    - For each 'INFO - Profile for <Stage>:' block, the parser scans up to the next
      stage header (or end-of-run) and extracts the 'CPU TOTAL' value.
    """
    # Get CPU clock
    if cpu_hz is None:
        m = re.search(r"Processor internal clock:\s*(\d+)\s*Hz", log_text)
        if m:
            cpu_hz = int(m.group(1))
        else:
            raise ValueError("CPU frequency not found in log and cpu_hz not provided.")

    runs: List[Dict[str, float]] = []

    # Find run boundaries
    run_starts = [m.start() for m in re.finditer(r"INFO - Running inference on image", log_text)]
    if not run_starts:
        return runs
    run_starts.append(len(log_text))  # sentinel

    for i in range(len(run_starts) - 1):
        chunk = log_text[run_starts[i]: run_starts[i+1]]
        run_dict: Dict[str, float] = {}

        # Stage headers within this run
        stage_iter = list(re.finditer(r"INFO - Profile for\s+([^:]+):", chunk))
        if not stage_iter:
            continue

        # End positions to slice each stage block
        ends = [m.start() for m in stage_iter[1:]] + [len(chunk)]

        for (m, end_pos) in zip(stage_iter, ends):
            stage = " ".join(m.group(1).split())
            block = chunk[m.end():end_pos]

            # Extract CPU TOTAL within this stage block
            cpu_match = re.search(r"CPU TOTAL:\s*(\d+)\s*cycles", block)
            if cpu_match:
                cycles = int(cpu_match.group(1))
                run_dict[stage] = cycles / float(cpu_hz) * 1000  # milliseconds

        if run_dict:
            runs.append(run_dict)

    return runs


def parse_perf_runs_native(log_text: str) -> List[Dict[str, float]]:
    """
    Parse logs and return a list of dicts (one per run), mapping stage -> seconds.
    A run starts at: 'INFO - Running inference on image ...'
    Stages are detected by:
      'INFO - Profile for <Stage>:'
      (followed later by) 'INFO - Duration: <num> microseconds'
    """
    runs: List[Dict[str, float]] = []

    # Find run boundaries
    run_starts = [m.start() for m in re.finditer(r"INFO - Running inference on image", log_text)]
    if not run_starts:
        return runs
    run_starts.append(len(log_text))  # sentinel

    for i in range(len(run_starts) - 1):
        chunk = log_text[run_starts[i]: run_starts[i+1]]
        run_dict: Dict[str, float] = {}

        # Extract (stage, duration_us) pairs inside this run
        for stage, us in re.findall(
            r"INFO - Profile for\s+([^:]+):.*?INFO - Duration:\s+(\d+)\s+microseconds",
            chunk,
            flags=re.DOTALL,
        ):
            stage = " ".join(stage.split())  # tidy spacing
            run_dict[stage] = int(us) / 1_000.0  # milliseconds

        if run_dict:
            runs.append(run_dict)

    return runs


def summarize_perf(runs: List[Dict[str, float]]) -> Dict[str, Dict[str, float]]:
    """
    Given output of parse_perf_runs, compute mean & std for each stage using numpy.
    Returns dict: {stage: {"mean": float, "std": float}}
    """
    stage_values: Dict[str, List[float]] = {}
    for run in runs:
        for stage, val in run.items():
            stage_values.setdefault(stage, []).append(val)

    summary: Dict[str, Dict[str, float]] = {}
    for stage, vals in stage_values.items():
        arr = np.array(vals, dtype=float)
        summary[stage] = {
            "mean": float(np.mean(arr)),
            "std": float(np.std(arr, ddof=1)) if len(arr) > 1 else 0.0
        }
    return summary

In [36]:
logs_native = """INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Aug 20 2025 @ 11:39:16
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added  support to op resolver
INFO - Creating allocator using tensor arena at 0x6094b9e8c030
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 4
Box: [183.85 30.64 61.28 61.28]. Position: [204.91 82.35]. Confidence: 0.80
Box: [183.85 91.92 91.92 30.64]. Position: [204.91 122.57]. Confidence: 0.80
Box: [122.57 153.21 61.28 91.92]. Position: [164.70 143.63]. Confidence: 0.80
Box: [91.92 0.00 61.28 91.92]. Position: [143.63 82.35]. Confidence: 0.61
INFO - Profile for CopyOutput:
INFO - Duration: 53 microseconds
INFO - Profile for Inference:
INFO - Duration: 744479 microseconds
INFO - Profile for ModelInit:
INFO - Duration: 1368 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 12 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 50 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 4
Box: [183.85 91.92 91.92 30.64]. Position: [204.91 122.57]. Confidence: 0.86
Box: [183.85 30.64 61.28 61.28]. Position: [185.76 82.35]. Confidence: 0.80
Box: [122.57 153.21 61.28 91.92]. Position: [164.70 143.63]. Confidence: 0.80
Box: [91.92 0.00 61.28 91.92]. Position: [143.63 82.35]. Confidence: 0.71
INFO - Profile for CopyOutput:
INFO - Duration: 26 microseconds
INFO - Profile for Inference:
INFO - Duration: 729481 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 10 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 36 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 5
Box: [183.85 30.64 61.28 61.28]. Position: [185.76 82.35]. Confidence: 0.80
Box: [183.85 91.92 91.92 30.64]. Position: [204.91 122.57]. Confidence: 0.80
Box: [122.57 153.21 61.28 91.92]. Position: [164.70 143.63]. Confidence: 0.80
Box: [91.92 0.00 61.28 91.92]. Position: [143.63 82.35]. Confidence: 0.71
Box: [45.96 76.60 45.96 30.64]. Position: [93.84 93.84]. Confidence: 0.20
INFO - Profile for CopyOutput:
INFO - Duration: 26 microseconds
INFO - Profile for Inference:
INFO - Duration: 726645 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 9 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 37 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: """


logs_alif = """[sudo] password for pglab3: 
picocom v3.1

port is        : /dev/ttyUSB0
flowcontrol    : none
baudrate is    : 921600
parity is      : none
databits are   : 8
stopbits are   : 1
escape is      : C-a
local echo is  : no
noinit is      : no
noreset is     : no
hangup is      : no
nolock is      : no
send_cmd is    : sz -vv
receive_cmd is : rz -vv -E
imap is        : 
omap is        : 
emap is        : crcrlf,delbs,
logfile is     : none
initstring     : none
exit_after is  : not set
exit is        : no

Type [C-a] [C-h] to see available commands
Terminal ready

Wakeup Reason (LPTIMER) Interrupt!

Reset Reason [0](UNKNOWN)
PointGrab Board Rev.A
WARN - Backup 4K SRAM is not valid -> initializing!
INFO - Processor internal clock: 160000000Hz
INFO - platform_init: complete
INFO - Ethos-U device initialised
INFO - Ethos-U version info:
INFO -  Arch:       v1.1.0
INFO -  Driver:     v0.16.0
INFO -  MACs/cc:    128
INFO -  Cmd stream: v0
INFO - Target system design: Ensemble
INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Aug 20 2025 @ 11:36:04
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added ethos-u support to op resolver
INFO - Creating allocator using tensor arena at 0x2000000
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 2
                                           Box: [245.13 122.57 122.57 61.28]. Position: [329.40 164.70]. Confidence: 0.80
                                                                                                                         Box: [524.73 306.41 122.57 61.28]. Position: [540.06 333.23]. Confidence: 0.20
    INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229708 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 29368140 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2218 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20296039 cycles
INFO - NPU TOTAL: 20298257 cycles
INFO - CPU TOTAL: 24276810 cycles
INFO - Profile for ModelInit:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 568850 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 79107 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 131114 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 1
                                           Box: [245.13 122.57 122.57 61.28]. Position: [329.40 164.70]. Confidence: 0.71
                                                                                                                         INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229078 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 27442056 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2114 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20295817 cycles
INFO - NPU TOTAL: 20297931 cycles
INFO - CPU TOTAL: 24275651 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 75691 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135414 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 2
                                           Box: [245.13 122.57 122.57 61.28]. Position: [329.40 164.70]. Confidence: 0.80
                                                                                                                         Box: [494.09 306.41 153.21 61.28]. Position: [559.21 333.23]. Confidence: 0.20
    INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229344 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 27695815 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2233 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8294573 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4329613 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 20297228 cycles
INFO - NPU TOTAL: 20299461 cycles
INFO - CPU TOTAL: 24276887 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 77335 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135515 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: """

In [ ]:
logs_native_nano = """INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Aug 20 2025 @ 18:05:26
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added  support to op resolver
INFO - Creating allocator using tensor arena at 0x5c6b010c6030
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 5
Box: [175.97 75.88 101.71 50.05]. Position: [224.40 101.71]. Confidence: 0.86
Box: [75.88 0.00 75.88 101.71]. Position: [142.07 80.72]. Confidence: 0.80
Box: [125.92 125.92 50.05 101.71]. Position: [163.05 163.05]. Confidence: 0.80
Box: [175.97 25.83 50.05 75.88]. Position: [203.41 61.35]. Confidence: 0.50
Box: [50.05 175.97 50.05 50.05]. Position: [101.71 203.41]. Confidence: 0.50
INFO - Profile for CopyOutput:
INFO - Duration: 59 microseconds
INFO - Profile for Inference:
INFO - Duration: 803755 microseconds
INFO - Profile for ModelInit:
INFO - Duration: 1400 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 11 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 54 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 6
Box: [175.97 75.88 101.71 50.05]. Position: [224.40 122.69]. Confidence: 0.86
Box: [75.88 0.00 75.88 101.71]. Position: [142.07 80.72]. Confidence: 0.80
Box: [125.92 125.92 50.05 101.71]. Position: [163.05 163.05]. Confidence: 0.80
Box: [175.97 25.83 50.05 75.88]. Position: [203.41 61.35]. Confidence: 0.61
Box: [318.03 279.29 0.00 38.75]. Position: [318.03 297.05]. Confidence: 0.39
Box: [140.45 177.58 38.75 62.96]. Position: [174.35 184.04]. Confidence: 0.29
INFO - Profile for CopyOutput:
INFO - Duration: 27 microseconds
INFO - Profile for Inference:
INFO - Duration: 790995 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 10 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 43 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: 1

INFO - Running inference on image 0 => img_320.bmp
Final results:
Total number of inferences: 5
Box: [175.97 75.88 101.71 50.05]. Position: [224.40 122.69]. Confidence: 0.86
Box: [75.88 0.00 75.88 101.71]. Position: [142.07 80.72]. Confidence: 0.80
Box: [125.92 125.92 50.05 101.71]. Position: [163.05 163.05]. Confidence: 0.80
Box: [175.97 25.83 50.05 75.88]. Position: [203.41 61.35]. Confidence: 0.61
Box: [140.45 177.58 38.75 62.96]. Position: [174.35 184.04]. Confidence: 0.29
INFO - Profile for CopyOutput:
INFO - Duration: 25 microseconds
INFO - Profile for Inference:
INFO - Duration: 797537 microseconds
INFO - Profile for PostProcessing:
INFO - Duration: 10 microseconds
INFO - Profile for PreProcessing:
INFO - Duration: 25 microseconds


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  Choice: """
  
logs_alif_nano = """[sudo] password for pglab3: 
picocom v3.1

port is        : /dev/ttyUSB0
flowcontrol    : none
baudrate is    : 921600
parity is      : none
databits are   : 8
stopbits are   : 1
escape is      : C-a
local echo is  : no
noinit is      : no
noreset is     : no
hangup is      : no
nolock is      : no
send_cmd is    : sz -vv
receive_cmd is : rz -vv -E
imap is        : 
omap is        : 
emap is        : crcrlf,delbs,
logfile is     : none
initstring     : none
exit_after is  : not set
exit is        : no

Type [C-a] [C-h] to see available commands
Terminal ready

Wakeup Reason (LPTIMER) Interrupt!

Reset Reason [0](UNKNOWN)
PointGrab Board Rev.A
WARN - Backup 4K SRAM is not valid -> initializing!
INFO - Processor internal clock: 160000000Hz
INFO - platform_init: complete
INFO - Ethos-U device initialised
INFO - Ethos-U version info:
INFO -  Arch:       v1.1.0
INFO -  Driver:     v0.16.0
INFO -  MACs/cc:    128
INFO -  Cmd stream: v0
INFO - Target system design: Ensemble
INFO - ARM ML Embedded Evaluation Kit
INFO - Version 23.08.0 Build date: Aug 20 2025 @ 18:12:08
INFO - Copyright 2021-2023 Arm Limited and/or its affiliates <open-source-office@arm.com>

INFO - Added ethos-u support to op resolver
INFO - Creating allocator using tensor arena at 0x2000000
INFO - Allocating tensors


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 4
                                           Box: [503.69 151.75 151.75 151.75]. Position: [529.52 203.41]. Confidence: 0.80
                                                                                                                          Box: [303.50 0.00 100.09 100.09]. Position: [364.85 80.72]. Confidence: 0.61
   Box: [503.69 303.50 151.75 51.66]. Position: [529.52 326.10]. Confidence: 0.61
                                                                                 Box: [100.09 303.50 100.09 151.75]. Position: [161.44 326.10]. Confidence: 0.61
                                                                                                                                                                INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 230032 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 29387472 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2247 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8677430 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4407659 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 21135652 cycles
INFO - NPU TOTAL: 21137899 cycles
INFO - CPU TOTAL: 25117230 cycles
INFO - Profile for ModelInit:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 567606 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 91699 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135927 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 4
                                           Box: [503.69 151.75 151.75 151.75]. Position: [529.52 203.41]. Confidence: 0.80
                                                                                                                          Box: [100.09 303.50 100.09 151.75]. Position: [161.44 326.10]. Confidence: 0.71
      Box: [303.50 0.00 100.09 100.09]. Position: [364.85 80.72]. Confidence: 0.61
                                                                                  Box: [503.69 303.50 151.75 51.66]. Position: [529.52 326.10]. Confidence: 0.61
                                                                                                                                                                INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 229227 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 25780418 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2265 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8677430 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4407659 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 21134526 cycles
INFO - NPU TOTAL: 21136791 cycles
INFO - CPU TOTAL: 25114179 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 89348 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 135872 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: 1

dump binary memory /tmp/cam_image0_Bayer_to_RGB_320p.tif 0x8083400 0x80BB8B0 
INFO - Running inference on image from camera
Final results:
              Total number of inferences: 4
                                           Box: [503.69 151.75 151.75 151.75]. Position: [529.52 203.41]. Confidence: 0.80
                                                                                                                          Box: [303.50 0.00 100.09 100.09]. Position: [364.85 80.72]. Confidence: 0.61
   Box: [503.69 303.50 151.75 51.66]. Position: [529.52 326.10]. Confidence: 0.61
                                                                                 Box: [100.09 303.50 100.09 151.75]. Position: [161.44 326.10]. Confidence: 0.61
                                                                                                                                                                INFO - Profile for CopyOutput:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 228665 cycles
INFO - Profile for GetImage:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 26845430 cycles
INFO - Profile for Inference:
INFO - NPU IDLE: 2336 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 8677430 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 4407659 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 21135894 cycles
INFO - NPU TOTAL: 21138230 cycles
INFO - CPU TOTAL: 25115409 cycles
INFO - Profile for PostProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 92490 cycles
INFO - Profile for PreProcessing:
INFO - NPU IDLE: 0 cycles
INFO - NPU AXI0_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU AXI0_WR_DATA_BEAT_WRITTEN: 0 beats
INFO - NPU AXI1_RD_DATA_BEAT_RECEIVED: 0 beats
INFO - NPU ACTIVE: 0 cycles
INFO - NPU TOTAL: 0 cycles
INFO - CPU TOTAL: 130142 cycles


User input required
Enter option number from:

  1. Run detection inference
  4. Show NN model info
  5. List ifm
  6. Run tracker
  7. Go to STOP3 power mode
  8. Test Ethernet
  9. Test Microphone
  10. Test Env Sensor
  11. Test EEPROM
  12. Test PIR
  13. Test WDTM
  15. Test LED

  14. RADIO CLI

  Choice: """

In [38]:
summary = summarize_perf(parse_perf_runs_native(logs_native))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,CopyOutput,0.035,0.016
1,Inference,733.535,9.583
2,ModelInit,1.368,0.000
3,PostProcessing,0.010,0.002
4,PreProcessing,0.041,0.008


In [39]:
summary = summarize_perf(parse_perf_runs_from_cycles(logs_alif))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,CopyOutput,1.434,0.002
1,GetImage,176.054,6.541
2,Inference,151.728,0.004
3,ModelInit,3.555,0.000
4,PostProcessing,0.484,0.011
5,PreProcessing,0.838,0.016


In [42]:
summary = summarize_perf(parse_perf_runs_native(logs_native_nano))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,CopyOutput,0.037,0.019
1,Inference,797.429,6.381
2,ModelInit,1.400,0.000
3,PostProcessing,0.010,0.001
4,PreProcessing,0.041,0.015


In [43]:
summary = summarize_perf(parse_perf_runs_from_cycles(logs_alif_nano))
pd.DataFrame.from_dict(summary, orient='index').reset_index().rename(columns={'index': 'Stage'}).round(3)

,Stage,mean,std
0,CopyOutput,1.433,0.004
1,GetImage,170.861,11.583
2,Inference,156.973,0.010
3,ModelInit,3.548,0.000
4,PostProcessing,0.570,0.010
5,PreProcessing,0.837,0.021
